# 0. Định nghĩa thư viện (Cài đặt môi trường nếu cần)

In [1]:
!pip install pyspark pandas openpyxl

# 1. Imports và Cấu hình log

In [2]:
import os
import re
import logging
import glob
import pandas as pd
from datetime import datetime
from typing import Optional

# PySpark core
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, IntegerType, FloatType, TimestampType
)
from pyspark.sql.window import Window

# ML
from pyspark.ml.recommendation import ALS, ALSModel
from pyspark.ml.feature import (
    HashingTF, IDF, Tokenizer,
    StringIndexer, IndexToString
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# ─────────────────────────────────────────────
# Cấu hình logging
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("PlaylistRecommender")

# 2. Định nghĩa Tham số & Đường dẫn

In [3]:
# CẦN SỬA: Giảm tham số rank, max_iter và memory xuống mức an toàn cho Kaggle
DATA_PATHS = [
    "/kaggle/input/datasets/js042710/second3t1k/data DM/data DM/",
    "/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA/",
]
output_dir  = "/kaggle/working/recommendations"
model_dir   = "/kaggle/working/als_model"
n_recs      = 20
rank        = 20       # Giảm từ 50 xuống 20 để tránh model quá lớn
max_iter    = 10       # Giảm từ 20 xuống 10
reg_param   = 0.1
app_name    = "MusicPlaylistRecommender"
memory      = "9g"     # Giảm driver memory xuống 8g thay vì 12g


# 3. Khởi tạo Spark Session

In [4]:
# CẦN SỬA: Thay thế toàn bộ đoạn khởi tạo biến spark
spark = (
    SparkSession.builder
    .appName(app_name)
    .master("local[2]") # Dùng 2 CPUs của Kaggle Kernel
    # Tối ưu phân bổ RAM (Drivers = 8GB, Executors = 3GB)
    .config("spark.driver.memory", memory)
    .config("spark.executor.memory", "3g")
    .config("spark.driver.maxResultSize", "1g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.default.parallelism", "4")
    
    # Ép dung lượng chừa lại cho tính toán (Execution) nhiều hơn Storage
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.2")
    
    # Bắt buộc Spark lưu file spill/shuffle ở thư mục được Kaggle cho phép ghi
    .config("spark.local.dir", "/kaggle/working/tmp")
    
    # [KỸ THUẬT TỐI ƯU SERIALIZE & TASK LỚN]
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.rpc.message.maxSize", "1024") 
    .config("spark.kryoserializer.buffer.max", "512m")
    
    # Tối ưu ALS & Adapter
    .config("spark.ml.als.blockSize", "2048")
    .config("spark.sql.files.maxPartitionBytes", "67108864")   # Chia nhỏ file data ~64MB
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    
    # Tắt lỗi báo cáo
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
logger.info("SparkSession khởi tạo thành công — Spark %s", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/02 10:43:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/02 10:43:07 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
2026-04-02 10:43:13 [INFO] PlaylistRecommender — SparkSession khởi tạo thành công — Spark 4.0.2


# 4. Đọc Dữ Liệu (Load Data)

In [5]:
RAW_SCHEMA = StructType([
    StructField("user_id",        StringType(),    True),
    StructField("timestamp",      StringType(),    True),   
    StructField("recording_msid", StringType(),    True),
    StructField("track_name",     StringType(),    True),
    StructField("artist_name",    StringType(),    True),
    StructField("release_name",   StringType(),    True),
])

file_format = "auto"
dfs = []
paths = DATA_PATHS

all_files = []
for p in paths:
    if os.path.isdir(p):
        files_in_dir = [os.path.join(p, f) for f in os.listdir(p) 
                        if os.path.isfile(os.path.join(p, f))]
        all_files.extend(files_in_dir)
    elif os.path.isfile(p):
        all_files.append(p)
    else:
        all_files.extend(glob.glob(p))

for file_path in all_files:
    ext = os.path.splitext(file_path)[1].lower()
    
    fmt = file_format
    if fmt == "auto":
        is_zip = False
        try:
            with open(file_path, "rb") as f:
                if f.read(4) == b"PK\x03\x04":
                    is_zip = True
        except Exception:
            pass
            
        if is_zip: fmt = "xlsx"
        elif ext == ".parquet": fmt = "parquet"
        elif ext in [".xlsx", ".xls"]: fmt = "xlsx"
        elif ext == ".csv": fmt = "csv"
        else: continue 

    logger.info("Đang xử lý [%s]: %s", fmt.upper(), os.path.basename(file_path))

    try:
        if fmt == "csv":
            sample_df = spark.read.option("header", "true").option("sep", "\t").csv(file_path).limit(1)
            actual_sep = "," if len(sample_df.columns) <= 1 else "\t"
            
            df = (spark.read
                  .option("header", "true")
                  .option("sep", actual_sep)
                  .schema(RAW_SCHEMA)
                  .csv(file_path))

        elif fmt == "xlsx":
            engine_kwargs = {}
            if is_zip and ext not in [".xlsx", ".xls"]:
                engine_kwargs["engine"] = "openpyxl"
                
            pdf = pd.read_excel(file_path, **engine_kwargs)
            df = spark.createDataFrame(pdf)
            
            for field in RAW_SCHEMA.fields:
                if field.name in df.columns:
                    df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))
                else:
                    df = df.withColumn(field.name, F.lit(None).cast(field.dataType))
            
            df = df.select([f.name for f in RAW_SCHEMA.fields])

        elif fmt == "parquet":
            df = spark.read.schema(RAW_SCHEMA).parquet(file_path)
        
        else:
            continue

        count = df.count()
        logger.info("  → Thành công: %s bản ghi", f"{count:,}")
        dfs.append(df)

    except Exception as exc:
        logger.warning("  ⚠ Lỗi xử lý %s: %s", os.path.basename(file_path), exc)

if not dfs:
    raise ValueError("Không đọc được dữ liệu hợp lệ từ các đường dẫn được cung cấp!")

raw_df = dfs[0]
for df in dfs[1:]:
    raw_df = raw_df.unionByName(df, allowMissingColumns=True)


# Thay vì sample ngẫu nhiên dòng, phương pháp chuẩn là Sample theo User (Bốc ngẫu nhiên 10% số lượng user, nhưng đã trúng user nào thì giữ lại toàn bộ lịch sử từ cũ đến mới của user đó).

# 1. Lập danh sách lọc lấy ngẫu nhiên 10% lượng User
sampled_users = raw_df.select("user_id").distinct().sample(fraction=0.1, seed=42)
# 2. Giữ lại toàn bộ dữ liệu nghe nhạc (cả cũ lẫn mới) của những User đã được chọn
raw_df = raw_df.join(sampled_users, on="user_id", how="inner")

#logger.info("Hoàn tất tải dữ liệu. Tổng cộng: %s bản ghi", f"{raw_df.count():,}")

2026-04-02 10:43:13 [INFO] PlaylistRecommender — Đang xử lý [CSV]: clean_16.02.2026.listens.csv
2026-04-02 10:43:20 [INFO] PlaylistRecommender —   → Thành công: 2,294,926 bản ghi
2026-04-02 10:43:20 [INFO] PlaylistRecommender — Đang xử lý [CSV]: clean_17.02.2026.listens.csv
2026-04-02 10:43:23 [INFO] PlaylistRecommender —   → Thành công: 2,120,457 bản ghi
2026-04-02 10:43:23 [INFO] PlaylistRecommender — Đang xử lý [CSV]: clean_12.02.2026.listens.csv
2026-04-02 10:43:25 [INFO] PlaylistRecommender —   → Thành công: 2,178,159 bản ghi
2026-04-02 10:43:25 [INFO] PlaylistRecommender — Đang xử lý [CSV]: clean_08.02.2026.listens.csv
2026-04-02 10:43:28 [INFO] PlaylistRecommender —   → Thành công: 3,261,685 bản ghi
2026-04-02 10:43:28 [INFO] PlaylistRecommender — Đang xử lý [CSV]: clean_07.02.2026.listens.csv
2026-04-02 10:43:30 [INFO] PlaylistRecommender —   → Thành công: 3,428,662 bản ghi
2026-04-02 10:43:31 [INFO] PlaylistRecommender — Đang xử lý [CSV]: clean_22.02.2026.listens.csv
2026-04-0

# 5. Làm sạch Dữ liệu (Clean Data)

In [6]:
from pyspark import StorageLevel
logger.info("Bắt đầu làm sạch dữ liệu...")

df = raw_df

# 1. Bỏ null ở cột bắt buộc
required_cols = ["user_id", "recording_msid", "track_name", "artist_name"]
df = df.dropna(subset=required_cols)

# 2. Chuẩn hóa chuỗi
for col in ["track_name", "artist_name", "release_name"]:
    df = df.withColumn(col, F.trim(F.lower(F.col(col))))

df = df.withColumn("user_id", F.trim(F.col("user_id")))
df = df.withColumn("recording_msid", F.trim(F.col("recording_msid")))

# 3. Parse timestamp
df = df.withColumn(
    "event_time",
    F.coalesce(
        F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("timestamp", "yyyy-MM-dd'T'HH:mm:ss"),
        F.to_timestamp("timestamp", "dd/MM/yyyy HH:mm:ss"),
        F.to_timestamp(F.from_unixtime(F.col("timestamp").cast(LongType()))),
    )
)

# 4. Trích xuất đặc trưng thời gian
df = (df
      .withColumn("hour_of_day", F.hour("event_time"))
      .withColumn("day_of_week", F.dayofweek("event_time"))
      .withColumn("month",       F.month("event_time")))

# 5. Loại bỏ trùng lặp
df = df.dropDuplicates(["user_id", "recording_msid", "timestamp"])
logger.info("Đã lọc xong dữ liệu trùng lặp (tắt chế độ count() để tối ưu tốc độ DAG).")

clean_df = df.persist(StorageLevel.MEMORY_AND_DISK)

2026-04-02 10:48:19 [INFO] PlaylistRecommender — Bắt đầu làm sạch dữ liệu...
2026-04-02 10:48:19 [INFO] PlaylistRecommender — Đã lọc xong dữ liệu trùng lặp (tắt chế độ count() để tối ưu tốc độ DAG).


# 6. Phân tích & Thống kê

In [7]:
logger.info("=" * 60)
logger.info("THỐNG KÊ DỮ LIỆU")
logger.info("=" * 60)

df_stat = clean_df

total      = df_stat.count()
#n_users_stat    = df_stat.select("user_id").distinct().count()
#n_tracks_stat   = df_stat.select("recording_msid").distinct().count()
#n_artists_stat  = df_stat.select("artist_name").distinct().count()

# Chuyển sang hàm Đếm ước lượng (Approximate Count) siêu tốc của Spark (sai số ~5% không RAM bằng số 0)
n_users_stat   = df_stat.agg(F.approx_count_distinct("user_id")).collect()[0][0]
n_tracks_stat  = df_stat.agg(F.approx_count_distinct("recording_msid")).collect()[0][0]
n_artists_stat = df_stat.agg(F.approx_count_distinct("artist_name")).collect()[0][0]


logger.info("Tổng số sự kiện    : %s", f"{total:,}")
logger.info("Số người dùng      : %s", f"{n_users_stat:,}")
logger.info("Số bài hát duy nhất: %s", f"{n_tracks_stat:,}")
logger.info("Số nghệ sĩ duy nhất: %s", f"{n_artists_stat:,}")

# Top 10 bài nghe nhiều nhất
logger.info("\nTop 10 bài hát phổ biến nhất:")
(df_stat.groupBy("track_name", "artist_name")
   .count()
   .orderBy(F.desc("count"))
   .limit(10)
   .show(truncate=False))

# Top 10 nghệ sĩ
logger.info("Top 10 nghệ sĩ phổ biến nhất:")
(df_stat.groupBy("artist_name")
   .agg(F.countDistinct("user_id").alias("n_listeners"),
        F.count("*").alias("total_plays"))
   .orderBy(F.desc("total_plays"))
   .limit(10)
   .show(truncate=False))

# Phân phối số bài nghe / người dùng
logger.info("Phân phối hoạt động người dùng:")
(df_stat.groupBy("user_id")
   .count()
   .select(
       F.min("count").alias("min_plays"),
       F.avg("count").alias("avg_plays"),
       F.max("count").alias("max_plays"),
       F.expr("percentile(count, 0.5)").alias("median_plays"),
       F.expr("percentile(count, 0.9)").alias("p90_plays"),
   )
   .show())

2026-04-02 10:48:19 [INFO] PlaylistRecommender — ============================================================
2026-04-02 10:48:19 [INFO] PlaylistRecommender — THỐNG KÊ DỮ LIỆU
2026-04-02 10:48:19 [INFO] PlaylistRecommender — ============================================================
26/04/02 10:50:32 WARN TaskSetManager: Stage 246 contains a task of very large size (9807 KiB). The maximum recommended task size is 1000 KiB.
26/04/02 10:53:14 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-04-02 10:53:22 [INFO] PlaylistRecommender — Tổng số sự kiện    : 16,421,630
2026-04-02 10:53:22 [INFO] PlaylistRecommender — Số người dùng      : 28,537
2026-04-02 10:53:22 [INFO] PlaylistRecommender — Số bài hát duy nhất: 4,853,242
2026-04-02 10:53:22 [INFO] PlaylistRecommender — Số nghệ sĩ duy nhất: 859,246
2026-04-02 10:53:22 [INFO] PlaylistRecommender — 
Top 10 bài hát p

+--------------------------+------------------+-----+
|track_name                |artist_name       |count|
+--------------------------+------------------+-----+
|pink noise (loopable)     |relaxation channel|3971 |
|birds and beaches         |the sound reserve |2837 |
|mr. brightside            |the killers       |2072 |
|in the end                |linkin park       |1993 |
|numb                      |linkin park       |1985 |
|the less i know the better|tame impala       |1977 |
|blinding lights           |the weeknd        |1836 |
|chop suey!                |system of a down  |1812 |
|creep                     |radiohead         |1748 |
|seven nation army         |the white stripes |1614 |
+--------------------------+------------------+-----+



2026-04-02 10:53:58 [INFO] PlaylistRecommender — Phân phối hoạt động người dùng:


+--------------+-----------+-----------+
|artist_name   |n_listeners|total_plays|
+--------------+-----------+-----------+
|radiohead     |3207       |47299      |
|taylor swift  |2177       |42901      |
|the beatles   |2603       |42370      |
|linkin park   |3362       |40943      |
|regina spektor|421        |35861      |
|pink floyd    |2483       |32263      |
|kanye west    |1895       |30895      |
|daft punk     |2609       |29138      |
|kendrick lamar|2212       |27555      |
|lady gaga     |1986       |26263      |
+--------------+-----------+-----------+

+---------+-----------------+---------+------------+------------------+
|min_plays|        avg_plays|max_plays|median_plays|         p90_plays|
+---------+-----------------+---------+------------+------------------+
|        1|636.6947115384615|   287884|        72.0|509.90000000000146|
+---------+-----------------+---------+------------+------------------+



# 7. Feature Engineering - Ma trận Tương tác

In [8]:
logger.info("Xây dựng ma trận tương tác user–track...")

# Tần suất nghe (Chỉ group theo ID để giảm shuffle write)
interaction = (clean_df
    .groupBy("user_id", "recording_msid")
    .agg(
        F.count("*").alias("play_count"),
        F.max("event_time").alias("last_played"),
    ))

# Recency weight: bài nghe gần đây được điểm cao hơn
max_ts = interaction.agg(F.max("last_played")).collect()[0][0]
if max_ts is None:
    max_ts = datetime.now()

interaction = interaction.withColumn(
    "days_since_last",
    F.datediff(F.lit(max_ts), F.col("last_played")).cast(FloatType())
)

# rating = log(1 + play_count) × decay(days)
interaction = interaction.withColumn(
    "rating",
    F.log1p(F.col("play_count").cast(FloatType())) *
    F.exp(F.col("days_since_last") * F.lit(-0.01))
)

# Chỉ giữ lại các tương tác của những cặp thỏa mãn tổng số lượt nghe >= 3 
interaction = interaction.filter(F.col("play_count") >= 3)

# THÊM LỆNH NÀY: Cache ma trận để StringIndexer không bị chạy lặp lại lệnh groupBy
interaction = interaction.cache()

logger.info("Ma trận tương tác: %s cặp (user, track)", f"{interaction.count():,}")


2026-04-02 10:54:02 [INFO] PlaylistRecommender — Xây dựng ma trận tương tác user–track...
2026-04-02 10:54:40 [INFO] PlaylistRecommender — Ma trận tương tác: 1,124,853 cặp (user, track)


# 8. Feature Engineering - Mã hóa ID tĩnh

In [9]:
from pyspark import StorageLevel
logger.info("Mã hóa ID người dùng và bài hát...")

user_indexer  = StringIndexer(inputCol="user_id",
                              outputCol="user_idx",
                              handleInvalid="keep")
track_indexer = StringIndexer(inputCol="recording_msid",
                              outputCol="track_idx",
                              handleInvalid="keep")

pipeline = Pipeline(stages=[user_indexer, track_indexer])
indexer_model = pipeline.fit(interaction)
encoded  = indexer_model.transform(interaction)

# ============ THÊM MỚI TỪ ĐÂY ============
# Phục hồi metadata bài hát (tiết kiệm tài nguyên do lấy sau khi interaction đã được filter)
track_info = clean_df.select("recording_msid", "track_name", "artist_name", "release_name").dropDuplicates(["recording_msid"])
encoded = encoded.join(track_info, on="recording_msid", how="left")
# =========================================

# Ép kiểu sang Integer (ALS yêu cầu)
encoded = (encoded
           .withColumn("user_idx",  F.col("user_idx").cast(IntegerType()))
           .withColumn("track_idx", F.col("track_idx").cast(IntegerType())))

encoded_df = encoded.persist(StorageLevel.MEMORY_AND_DISK)


2026-04-02 10:54:40 [INFO] PlaylistRecommender — Mã hóa ID người dùng và bài hát...


# 9. Collaborative Filtering (Huấn luyện Mô hình ALS)

In [10]:
alpha_param = 40.0
seed_val = 42
logger.info("Huấn luyện ALS — rank=%d, iter=%d, reg=%.3f", rank, max_iter, reg_param)

train_df, test_df = encoded_df.randomSplit([0.8, 0.2], seed=seed_val)

als = ALS(
    rank             = rank,
    maxIter          = max_iter,
    regParam         = reg_param,
    alpha            = alpha_param,
    userCol          = "user_idx",
    itemCol          = "track_idx",
    ratingCol        = "rating",
    coldStartStrategy= "drop",      
    implicitPrefs    = True,        
    seed             = seed_val,
)

als_model = als.fit(train_df)

# Đánh giá trên tập test
predictions = als_model.transform(test_df).dropna(subset=["prediction"])
evaluator   = RegressionEvaluator(
    metricName   = "rmse",
    labelCol     = "rating",
    predictionCol= "prediction",
)
rmse = evaluator.evaluate(predictions)
logger.info("ALS huấn luyện xong — RMSE: %.4f", rmse)

# Lưu model
als_model.write().overwrite().save(model_dir)
logger.info("Mô hình ALS đã được lưu tại: %s", model_dir)

2026-04-02 10:54:47 [INFO] PlaylistRecommender — Huấn luyện ALS — rank=20, iter=10, reg=0.100
26/04/02 10:54:50 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:55:31 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:06 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:10 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:30 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:34 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:40 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:43 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:47 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:52 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 10:56:56 WARN DAGScheduler:

# 10. Cold-start Problem — Xử lý User/Track Mới
Khi có **User mới** (chưa có lịch sử nghe) hoặc **bài hát mới** (chưa xuất hiện trong ma trận ALS),
mô hình ALS không thể dự đoán. Hai chiến lược fallback:
- **Fallback A:** Gợi ý **Top Trending** toàn hệ thống (bài phổ biến nhất).
- **Fallback B:** Nếu biết user vừa nghe bài nào đó → gợi ý bài **tương tự nội dung (TF-IDF cosine similarity)**.

In [11]:
# ── Bước 1: Xây dựng TF-IDF vector cho từng bài hát ──────────────────
logger.info("Xây dựng đặc trưng nội dung TF-IDF...")

track_meta = (clean_df
    .select("recording_msid", "track_name", "artist_name", "release_name")
    .dropDuplicates(["recording_msid"])
    .withColumn(
        "content_text",
        F.concat_ws(" ",
            F.col("track_name"),
            F.col("artist_name"),
            F.col("release_name"))
    ))

tokenizer   = Tokenizer(inputCol="content_text", outputCol="words")
hashing_tf  = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=10000)
idf         = IDF(inputCol="raw_features", outputCol="tfidf_features", minDocFreq=2)

tfidf_pipeline = Pipeline(stages=[tokenizer, hashing_tf, idf])
tfidf_model    = tfidf_pipeline.fit(track_meta)
track_features = tfidf_model.transform(track_meta)
track_features = track_features.select(
    "recording_msid", "track_name", "artist_name", "release_name", "tfidf_features"
).cache()

#logger.info("TF-IDF features: %s bài hát", f"{track_features.count():,}")

# ── Bước 2: Top Trending toàn hệ thống (Fallback A) ──────────────────
logger.info("Tính Top Trending toàn hệ thống...")
top_trending = (clean_df
    .groupBy("recording_msid", "track_name", "artist_name", "release_name")
    .agg(
        F.count("*").alias("total_plays"),
        F.approx_count_distinct("user_id").alias("unique_listeners"),  # <--- BÍ QUYẾT LÀ Ở ĐÂY
    )
    .withColumn(
        "trending_score",
        F.col("total_plays").cast(FloatType()) * F.lit(0.6) +
        F.col("unique_listeners").cast(FloatType()) * F.lit(0.4)
    )
    .orderBy(F.desc("trending_score"))
    .limit(200)
    .cache())

logger.info("Top Trending sẵn sàng! Top 10:")
top_trending.select("track_name", "artist_name", "total_plays", "unique_listeners").show(10, truncate=False)

2026-04-02 11:01:39 [INFO] PlaylistRecommender — Xây dựng đặc trưng nội dung TF-IDF...
2026-04-02 11:02:39 [INFO] PlaylistRecommender — Tính Top Trending toàn hệ thống...
2026-04-02 11:02:39 [INFO] PlaylistRecommender — Top Trending sẵn sàng! Top 10:


+--------------------------+------------------+-----------+----------------+
|track_name                |artist_name       |total_plays|unique_listeners|
+--------------------------+------------------+-----------+----------------+
|pink noise (loopable)     |relaxation channel|3971       |1               |
|birds and beaches         |the sound reserve |2837       |1               |
|the less i know the better|tame impala       |1917       |702             |
|chop suey!                |system of a down  |1673       |608             |
|mr. brightside            |the killers       |1461       |574             |
|creep                     |radiohead         |1436       |568             |
|seven nation army         |the white stripes |1414       |550             |
|take me out               |franz ferdinand   |1347       |510             |
|toxicity                  |system of a down  |1315       |541             |
|do i wanna know?          |arctic monkeys    |1319       |518             |

In [12]:
# ── Fallback A: Gợi ý Top Trending ───────────────────────────────────
def recommend_cold_start_trending(n=20):
    """
    Trả về top-N bài hát phổ biến nhất trong toàn hệ thống.
    Dùng khi user_id không tồn tại trong ALS model (User mới).
    """
    result = (top_trending
        .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("trending_score"))))
        .filter(F.col("rank") <= n)
        .select("rank", "track_name", "artist_name", "release_name", "total_plays", "trending_score"))
    return result


# ── Fallback B: Gợi ý bài tương tự (TF-IDF Cosine Similarity) ────────
def recommend_similar_tracks(seed_recording_msid: str, n: int = 10, max_candidates: int = 5000):
    """
    Gợi ý n bài hát có nội dung gần giống nhất với bài hát đầu vào.
    
    [TỐI ƯU CHO KAGGLE]: Để tránh OOM Crash, chúng ta chỉ quét trên một `max_candidates` 
    (tệp con ngẫu nhiên) thay vì quét hàng triệu bản ghi bằng UDF.
    """
    # Lấy vector TF-IDF của bài gốc
    seed_row = track_features.filter(F.col("recording_msid") == seed_recording_msid).first()
    if seed_row is None:
        logger.warning("Không tìm thấy recording_msid=%s!", seed_recording_msid)
        return None

    seed_vec  = seed_row["tfidf_features"]
    seed_name = seed_row["track_name"]
    logger.info("Tìm bài tương tự cho: '%s' (msid=%s)", seed_name, seed_recording_msid)

    # UDF tính Cosine Similarity
    def cosine_sim(vec):
        if vec is None:
            return 0.0
        dot    = float(seed_vec.dot(vec))
        norm_a = float(seed_vec.norm(2))
        norm_b = float(vec.norm(2))
        if norm_a == 0 or norm_b == 0:
            return 0.0
        return dot / (norm_a * norm_b)

    cosine_udf = F.udf(cosine_sim, FloatType())
    
    # Lọc bỏ bài gốc và lấy random 5000 bài để tìm hàng xóm gần nhất
    candidates = (track_features
        .filter(F.col("recording_msid") != seed_recording_msid)
        .sample(fraction=0.2, seed=42)  # Lấy ngẫu nhiên ~20% dữ liệu
        .limit(max_candidates)          # Nắp hộp giới hạn tối đa 5000 bài
    )
    
    # Tính similarity trên candidates đã thu nhỏ
    similar = (candidates
        .withColumn("similarity", cosine_udf(F.col("tfidf_features")))
        .orderBy(F.desc("similarity"))
        .limit(n)
        .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("similarity"))))
        .select("rank", "track_name", "artist_name", "release_name", "similarity")
    )

    return similar



# ── Demo cả 2 Fallback ───────────────────────────────────────────────
print("=" * 65)
print("[FALLBACK A] TOP TRENDING — Dành cho User mới chưa có lịch sử:")
print("=" * 65)
recommend_cold_start_trending(n=10).show(10, truncate=False)

print("\n" + "=" * 65)
print("[FALLBACK B] SIMILAR TRACKS — Gợi ý bài tương tự nội dung:")
print("=" * 65)
# Tự động lấy bài đầu tiên trong dữ liệu làm ví dụ
first_track = track_features.first()
example_seed = first_track["recording_msid"]
example_name = first_track["track_name"]
print(f"  → Bài gốc (seed): '{example_name}' | msid: {example_seed}")
similar_result = recommend_similar_tracks(seed_recording_msid=example_seed, n=10)
if similar_result:
    similar_result.show(10, truncate=False)

[FALLBACK A] TOP TRENDING — Dành cho User mới chưa có lịch sử:


26/04/02 11:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:03:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+--------------------------+------------------+---------------------------------------------------+-----------+------------------+
|rank|track_name                |artist_name       |release_name                                       |total_plays|trending_score    |
+----+--------------------------+------------------+---------------------------------------------------+-----------+------------------+
|1   |pink noise (loopable)     |relaxation channel|pink noise (loopable)                              |3971       |2383.0            |
|2   |birds and beaches         |the sound reserve |30 serene ocean wave tracks: music for deep sleepng|2837       |1702.6000000000001|
|3   |the less i know the better|tame impala       |currents                                           |1917       |1431.0            |
|4   |chop suey!                |system of a down  |toxicity                                           |1673       |1247.0            |
|5   |mr. brightside            |the killers    

2026-04-02 11:04:26 [INFO] PlaylistRecommender — Tìm bài tương tự cho: 'sky-high bridge' (msid=000010c4-944d-4177-8202-78c930df49d7)


  → Bài gốc (seed): 'sky-high bridge' | msid: 000010c4-944d-4177-8202-78c930df49d7


26/04/02 11:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:04:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----+--------------------------------------+---------------------------+--------------------------------------------+----------+
|rank|track_name                            |artist_name                |release_name                                |similarity|
+----+--------------------------------------+---------------------------+--------------------------------------------+----------+
|1   |lady's bridge                         |richard hawley             |lady's bridge                               |0.2182608 |
|2   |spacemen - remix                      |total recall, vaski        |time to blackout                            |0.19949225|
|3   |materials recording the fibres of time|grumbling fur              |preternaturals                              |0.18881238|
|4   |broken feat.peter jones               |barock project             |pt.2 overture                               |0.18658017|
|5   |things'll never be the same           |spacemen 3                 |forged prescripti

26/04/02 11:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:04:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


# 11. Gợi ý hàng loạt (Cho tất cả user)

In [13]:
logger.info("Batch recommendation — %d users × %d tracks", -1, n_recs)

user_subset = encoded_df.select("user_idx").distinct()
batch_recs = als_model.recommendForUserSubset(user_subset, n_recs)

# Giải nén + join metadata
flat = (batch_recs
    .withColumn("rec", F.explode("recommendations"))
    .select(
        "user_idx",
        F.col("rec.track_idx").alias("track_idx"),
        F.col("rec.rating").alias("als_score"),
    ))

track_map = (encoded_df
    .select("track_idx", "recording_msid",
            "track_name", "artist_name", "release_name")
    .dropDuplicates(["track_idx"]))

batch_result = (flat
    .join(track_map, on="track_idx", how="left")
    .withColumn("rank",
        F.row_number().over(
            Window.partitionBy("user_idx").orderBy(F.desc("als_score"))))
    .filter(F.col("rank") <= n_recs)
    .select("user_idx", "rank", "track_name",
            "artist_name", "release_name", "als_score"))

# Lưu Batch Results
out_batch = f"{output_dir}/batch_recommendations"
logger.info("Lưu batch kết quả → %s [PARQUET]", out_batch)
writer = batch_result.coalesce(10).write.mode("overwrite")
writer.parquet(out_batch)
logger.info("Lưu thành công!")

2026-04-02 11:04:50 [INFO] PlaylistRecommender — Batch recommendation — -1 users × 20 tracks
2026-04-02 11:04:51 [INFO] PlaylistRecommender — Lưu batch kết quả → /kaggle/working/recommendations/batch_recommendations [PARQUET]
26/04/02 11:04:52 WARN DAGScheduler: Broadcasting large task binary with size 42.1 MiB
26/04/02 11:04:54 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 11:04:55 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 11:05:35 WARN DAGScheduler: Broadcasting large task binary with size 42.1 MiB
26/04/02 11:05:37 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/04/02 11:07:10 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/04/02 11:07:15 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/04/02 11:07:20 WARN DAGScheduler: Broadcasting large task binary with size 42.4 MiB
2026-04-02 11:07:23 [INFO] PlaylistRecommender — Lưu thành công!


# 12. Gợi ý cho 1 User Cụ Thể (Kèm Lọc bài đã nghe & Tăng trọng số nghệ sĩ)

In [14]:
sample_user = None
if sample_user is None:
    sample_user = (encoded_df
        .groupBy("user_id")
        .count()
        .orderBy(F.desc("count"))
        .first()["user_id"])
    logger.info("Chọn user demo (hoạt động nhất): %s", sample_user)

user_id = sample_user
exclude_heard = True
logger.info("Tạo gợi ý cho user: %s (top %d)", user_id, n_recs)

user_row = (encoded_df
            .filter(F.col("user_id") == user_id)
            .select("user_idx")
            .first())

if user_row is None:
    logger.warning("Không tìm thấy user_id=%s trong dữ liệu!", user_id)
else:
    user_idx = user_row["user_idx"]

    # 1. ALS gợi ý top-N×5 
    user_df  = spark.createDataFrame([(int(user_idx),)], ["user_idx"])
    recs     = als_model.recommendForUserSubset(user_df, n_recs * 5)

    recs_flat = (recs
        .withColumn("rec", F.explode("recommendations"))
        .select(
            F.col("rec.track_idx").alias("track_idx"),
            F.col("rec.rating").alias("als_score"),
        ))

    # 2. Decode track_idx → recording_msid
    track_map = (encoded_df
        .select("track_idx", "recording_msid",
                "track_name", "artist_name", "release_name")
        .dropDuplicates(["track_idx"]))

    recs_info = recs_flat.join(track_map, on="track_idx", how="left")

    # 3. Loại bỏ bài đã nghe 
    if exclude_heard:
        heard = (encoded_df
            .filter(F.col("user_id") == user_id)
            .select("recording_msid")
            .distinct())
        recs_info = recs_info.join(heard, on="recording_msid", how="left_anti")

    # 4. Content bonus — nghệ sĩ yêu thích
    fav_artists = (encoded_df
        .filter(F.col("user_id") == user_id)
        .groupBy("artist_name")
        .agg(F.sum("rating").alias("artist_affinity"))
        .orderBy(F.desc("artist_affinity"))
        .limit(10))

    recs_info = recs_info.join(fav_artists, on="artist_name", how="left")
    recs_info = recs_info.fillna({"artist_affinity": 0.0})

    # 5. Tính hybrid_score
    max_als = recs_info.agg(F.max("als_score")).collect()[0][0] or 1.0
    max_aff = fav_artists.agg(F.max("artist_affinity")).collect()[0][0] or 1.0

    recs_info = recs_info.withColumn(
        "hybrid_score",
        (F.col("als_score") / F.lit(max_als)) * F.lit(0.7) +
        (F.col("artist_affinity") / F.lit(max_aff)) * F.lit(0.3)
    )

    # 6. Sắp xếp và lấy top-N
    window_rank = Window.orderBy(F.desc("hybrid_score"))
    playlist = (recs_info
        .withColumn("rank", F.row_number().over(window_rank))
        .filter(F.col("rank") <= n_recs)
        .select("rank", "track_name", "artist_name",
                "release_name", "hybrid_score")
        .orderBy("rank"))

    logger.info("Đã tạo playlist %d bài cho %s", playlist.count(), user_id)

    logger.info("\n🎵 PLAYLIST GỢI Ý CHO USER: %s", user_id)
    logger.info("─" * 60)
    playlist.show(n_recs, truncate=False)

    # Lưu kết quả file CSV
    out_csv = f"{output_dir}/sample_playlist_{str(user_id)}"
    logger.info("Lưu playlist → %s [CSV]", out_csv)
    writer = playlist.coalesce(10).write.mode("overwrite")
    writer.option("header", "true").csv(out_csv)
    logger.info("Lưu thành công!")

26/04/02 11:07:24 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
2026-04-02 11:07:43 [INFO] PlaylistRecommender — Chọn user demo (hoạt động nhất): 21830
2026-04-02 11:07:43 [INFO] PlaylistRecommender — Tạo gợi ý cho user: 21830 (top 20)
26/04/02 11:07:44 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 11:07:46 WARN DAGScheduler: Broadcasting large task binary with size 42.1 MiB
26/04/02 11:07:47 WARN DAGScheduler: Broadcasting large task binary with size 42.0 MiB
26/04/02 11:07:56 WARN DAGScheduler: Broadcasting large task binary with size 42.1 MiB
26/04/02 11:08:10 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/04/02 11:08:15 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/04/02 11:08:18 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/04/02 11:08:23 WARN DAGScheduler: Broadcasting large task binary with size 42.2 MiB
26/04/02 11:08:42 WARN DAGScheduler: Broadcas

+----+--------------------------+------------------+---------------------------------------------------------------+------------------+
|rank|track_name                |artist_name       |release_name                                                   |hybrid_score      |
+----+--------------------------+------------------+---------------------------------------------------------------+------------------+
|1   |theme from jets 'n' guns 2|machinae supremacy|jets 'n' guns 2 (original game soundtrack)                     |0.7               |
|2   |one by one                |h.e.a.t           |h.e.a.t ii                                                     |0.6597830779027967|
|3   |chaos theory              |amaranthe         |chaos theory                                                   |0.6509890878095306|
|4   |goatlord                  |goat              |world music                                                    |0.6505307637515422|
|5   |impurities                |le sserafim    

26/04/02 11:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 11:11:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/02 1

# 13. Dừng Spark

In [15]:
spark.stop()

In [16]:
!zip -r batch_recs.zip /kaggle/working/recommendations/batch_recommendations


IOStream.flush timed out
  adding: kaggle/working/recommendations/batch_recommendations/ (stored 0%)
  adding: kaggle/working/recommendations/batch_recommendations/._SUCCESS.crc (stored 0%)
  adding: kaggle/working/recommendations/batch_recommendations/.part-00003-2507bac9-5cea-4f78-8590-b4077de4a430-c000.snappy.parquet.crc (stored 0%)
  adding: kaggle/working/recommendations/batch_recommendations/part-00003-2507bac9-5cea-4f78-8590-b4077de4a430-c000.snappy.parquet (deflated 10%)
  adding: kaggle/working/recommendations/batch_recommendations/.part-00002-2507bac9-5cea-4f78-8590-b4077de4a430-c000.snappy.parquet.crc (stored 0%)
  adding: kaggle/working/recommendations/batch_recommendations/.part-00000-2507bac9-5cea-4f78-8590-b4077de4a430-c000.snappy.parquet.crc (stored 0%)
  adding: kaggle/working/recommendations/batch_recommendations/.part-00001-2507bac9-5cea-4f78-8590-b4077de4a430-c000.snappy.parquet.crc (stored 0%)
  adding: kaggle/working/recommendations/batch_recommendations/part-0000